In [21]:
import numpy as np
import pandas as pd
from glob import glob
import networkx as nx
import matplotlib.pyplot as plt
import ipycytoscape

from pyvis.network import Network
from IPython.display import IFrame
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import re

DATA_RAW_FILE = 'DATA_RAW_CLEANED_20250401_20251231_ANTOFAGASTA_8834.parquet'

CLEANING_DATA = False
SAMPLING = '24h'

DATA_RAW_FILE = [x for x in glob('../**', recursive=True) if DATA_RAW_FILE in x][0]
DATA_RAW_FILE

'..\\DataCleaning\\DATA_RAW_CLEANED_20250401_20251231_ANTOFAGASTA_8834.parquet'

## READING DATA RAW

### CLEANING FUNCTIONS

In [22]:
def replace_outliers_with_nan_kmeans(series, n_clusters=3, contamination=0.05, random_state=42):

    if not isinstance(series, pd.Series):
        raise TypeError("Input must be a pandas Series.")

    valid_data = series.dropna()
    
    if len(valid_data) == 0:
        return series 
        
    X = valid_data.values.reshape(-1, 1)

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    kmeans.fit(X)

    all_distances = kmeans.transform(X)
    dist_to_center = np.min(all_distances, axis=1)

    threshold = np.quantile(dist_to_center, 1 - contamination)
    outlier_mask_valid = dist_to_center > threshold
    outlier_indices = valid_data.index[outlier_mask_valid]

    series_clean = series.copy()
    
    if not pd.api.types.is_float_dtype(series_clean):
        series_clean = series_clean.astype(float)
        
    series_clean.loc[outlier_indices] = np.nan

    return series_clean

def fix_missing_points(series, limit=4):
    return series.interpolate(method='linear', limit=limit, limit_direction='forward')

def fill_nan_with_time_mean(series):

    if not isinstance(series, pd.Series):
        raise TypeError("Input must be a pandas Series.")
        
    filled_series = series.copy()

    if not isinstance(filled_series.index, pd.DatetimeIndex):
        try:
            filled_series.index = pd.to_datetime(filled_series.index)
        except Exception as e:
            raise ValueError("Series index must be convertible to DatetimeIndex.") from e

    time_means = filled_series.groupby([filled_series.index.hour, filled_series.index.minute]).transform('mean')

    filled_series = filled_series.fillna(time_means)

    return filled_series





In [23]:
DATA_RAW = pd.read_parquet(DATA_RAW_FILE)

if CLEANING_DATA:
    ## CONVERTING TO 1 HOUR SAMPLE


    print("Aplying outlier replacement")
    DATA_RAW = DATA_RAW.apply(lambda x: replace_outliers_with_nan_kmeans(x))

    print("Aplying fix missing points")
    DATA_RAW = DATA_RAW.apply(lambda x: fix_missing_points(x))

    print("Aplying fill nan with time mean")
    DATA_RAW = DATA_RAW.apply(lambda x: fill_nan_with_time_mean(x))

    DATA_RAW = DATA_RAW.T

DATA_RAW = DATA_RAW.T.resample(SAMPLING).max().T
DATA_RAW = DATA_RAW[DATA_RAW.mean(axis=1) > 1]

#Replacing zeros with values 
DATA_RAW = DATA_RAW.mask(DATA_RAW < 1)     #.replace(0, np.nan)
DATA_RAW = DATA_RAW.T.apply(fill_nan_with_time_mean).T



NODES_ID = pd.DataFrame( list(set(DATA_RAW.index.get_level_values(0).tolist() + DATA_RAW.index.get_level_values(1).tolist())) ).rename(columns={0: 'node'})
EDGES_ID = pd.DataFrame([x for x in range(1000, 1000+len(DATA_RAW))], index=DATA_RAW.index, columns=['ID'])
EDGES_ID_DICT = {k:v.ID for k,v in EDGES_ID.iterrows()}
EDGES_I_INV_DICT = {v.ID:k for k,v in EDGES_ID.iterrows()}


DATA_RAW

2025-04-01  \
A                                   B                                              
910c_afs_camar_oriente_r6_csr_02605 02_605                            125.449028   
                                    910c_rio_lauca_csr_r6_02045       618.785173   
910c_aguas_anf_balm_csr_r5_02413    02_413                            278.583791   
910c_aguas_anf_ctro_csr_r2_02408    02_408                            255.551094   
                                    910c_av_arg_toyota_csr_r2_02323    35.758373   
...                                                                          ...   
ne8000m14_antofagasta_hr_b_2001     ne8000m14_antofagasta_hr_a_2001  6438.459476   
sara_3_brigada_acrzda_02521         02_521                            148.197902   
sara_anf_av_bllvst_r3_csr_02611     02_611                            107.765217   
                                    910c_antofa_paihuano_csr_02469     84.074808   
                                    950d_antofagasta_mr_a_2001        170.343385   

                                                                      2025-04-02  \
A                                   B                                              
910c_afs_camar_oriente_r6_csr_02605 02_605                            125.437662   
                                    910c_rio_lauca_csr_r6_02045       639.214664   
910c_aguas_anf_balm_csr_r5_02413    02_413                            270.473974   
910c_aguas_anf_ctro_csr_r2_02408    02_408                            255.576680   
                                    910c_av_arg_toyota_csr_r2_02323    37.416077   
...                                                                          ...   
ne8000m14_antofagasta_hr_b_2001     ne8000m14_antofagasta_hr_a_2001  6442.106283   
sara_3_brigada_acrzda_02521         02_521                            133.349759   
sara_anf_av_bllvst_r3_csr_02611     02_611                            107.797893   
                                    910c_antofa_paihuano_csr_02469     84.074803   
                                    950d_antofagasta_mr_a_2001        170.343368   

                                                                      2025-04-03  \
A                                   B                                              
910c_afs_camar_oriente_r6_csr_02605 02_605                            125.469393   
                                    910c_rio_lauca_csr_r6_02045       682.033338   
910c_aguas_anf_balm_csr_r5_02413    02_413                            280.239047   
910c_aguas_anf_ctro_csr_r2_02408    02_408                            255.597140   
                                    910c_av_arg_toyota_csr_r2_02323    34.308665   
...                                                                          ...   
ne8000m14_antofagasta_hr_b_2001     ne8000m14_antofagasta_hr_a_2001  6438.605248   
sara_3_brigada_acrzda_02521         02_521                            151.804683   
sara_anf_av_bllvst_r3_csr_02611     02_611                            111.994382   
                                    910c_antofa_paihuano_csr_02469     84.074809   
                                    950d_antofagasta_mr_a_2001        170.343381   

                                                                      2025-04-04  \
A                                   B                                              
910c_afs_camar_oriente_r6_csr_02605 02_605                            125.460549   
                                    910c_rio_lauca_csr_r6_02045       674.378151   
910c_aguas_anf_balm_csr_r5_02413    02_413                            278.351314   
910c_aguas_anf_ctro_csr_r2_02408    02_408                            255.588761   
                                    910c_av_arg_toyota_csr_r2_02323    38.722708   
...                                                                          ...   
ne8000m14_antofagasta_hr_b_2001     ne8000m14_antofagasta_hr_a_2001  6442.145542   
sara_3_brigada_acrzda_02521        

In [24]:
def draw_nodes_pyvis(data_raw: pd.DataFrame, filename: str = 'graph.html', directed: bool = True, sample_time: bool = True):
    # 1. Prepare Data (Same logic as before)
    if not sample_time:
        #data = data_raw.mean(axis=1).to_frame().round(0).rename(columns={0: 'bw'})
        data = data_raw.loc[:,data_raw.sum().idxmax()].to_frame().round(0).rename(columns={0: 'bw'})
    else:
        _sample_date =  data_raw.T.sample(1).index.values[0]
        data = data_raw[_sample_date].to_frame().round(0).rename(columns={_sample_date: 'bw'})
    
    # Node Data
    nodes_list = list(set(data.index.get_level_values(0).astype(str).tolist() + 
                          data.index.get_level_values(1).astype(str).tolist()))
    nodes_data = pd.DataFrame(nodes_list, columns=["id"])
    
    # Assign Colors/Sizes
    nodes_data['color'] = 'blue'
    nodes_data['size'] = 15  # Pyvis uses different scaling, smaller is better
    
    # Red Nodes
    hr_mask = nodes_data['id'].str.match(r'.*hr.*')
    nodes_data.loc[hr_mask, 'color'] = 'red'
    nodes_data.loc[hr_mask, 'size'] = 30

    # Yellow Nodes
    mr_mask = nodes_data['id'].str.match(r'.*mr_.*')
    nodes_data.loc[mr_mask, 'color'] = '#FFD700' # Hex for yellow often works better
    nodes_data.loc[mr_mask, 'size'] = 25
    
    # Green Nodes
    id_mask = nodes_data['id'].str.match(r'\d+[isp]?_\d+$')
    nodes_data.loc[id_mask, 'color'] = 'green'
    nodes_data.loc[id_mask, 'size'] = 10

    nodes_dict = nodes_data.to_dict('records')
    
    # Edge Data
    edge_df = data.reset_index()
    edge_df.iloc[:, 0] = edge_df.iloc[:, 0].astype(str)
    edge_df.iloc[:, 1] = edge_df.iloc[:, 1].astype(str)
    edge_data = list(edge_df.itertuples(index=False, name=None))

    # 2. Build Network
    # cdn_resources='remote' ensures it works without local JS files
    net = Network(height='600px', width='100%', notebook=True, cdn_resources='remote', directed=directed) 
                  
    
    for node in nodes_dict:
        net.add_node(node['id'], color=node['color'], size=node['size'], label=node['id'])

    for src, tgt, bw in edge_data:
        net.add_edge(
            src, 
            tgt, 
            title=f"Bandwidth: {bw}",  # Shows on hover
            label=str(int(bw)),             # Shows permanently on the line
            width=2,
            font={'align': 'middle', 'strokeWidth': 0, 'background': 'white'} # Makes text easier to read
        )

    # 3. Physics Options (Similar to 'cose')
    net.force_atlas_2based(gravity=-50, central_gravity=0.01, spring_length=100, spring_strength=0.08)
    
    # 4. Save and Show
    # This writes a file 'graph.html' to your folder. Open this file to see the graph!
    net.show(filename) 
    return f"Graph saved to '{filename}'. Open this file to view."


draw_nodes_pyvis(DATA_RAW, filename='graph_year.html', sample_time=True)


graph_year.html


"Graph saved to 'graph_year.html'. Open this file to view."

# GMAN - Graph Neural Networks for Time Series Forecasting
### CONVERTING EDGES INTO SENSORS

In [25]:

DATA_RAW_MODEL = DATA_RAW.copy()
DATA_RAW_MODEL.index = [EDGES_ID_DICT[x] for x in DATA_RAW_MODEL.index]
DATA_RAW_MODEL = DATA_RAW_MODEL.T
DATA_RAW_MODEL = DATA_RAW_MODEL.round(2)
DATA_RAW_MODEL

,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,...,1112,1113,1114,1115,1116,1117,1118,1119,1120,1121
2025-04-01,125.45,618.79,278.58,255.55,35.76,78.79,261.73,204.90,334.08,121.34,...,1155.42,648.97,824.77,5379.25,976.85,6438.46,148.20,107.77,84.07,170.34
2025-04-02,125.44,639.21,270.47,255.58,37.42,78.79,303.56,197.74,319.94,127.48,...,1144.28,647.16,824.00,4627.40,1022.72,6442.11,133.35,107.80,84.07,170.34
2025-04-03,125.47,682.03,280.24,255.60,34.31,78.79,302.89,197.79,325.32,121.28,...,1090.09,646.88,860.68,5175.81,975.59,6438.61,151.80,111.99,84.07,170.34
2025-04-04,125.46,674.38,278.35,255.59,38.72,78.79,301.48,197.76,332.32,121.29,...,783.38,646.57,823.21,5054.96,977.53,6442.15,141.84,107.83,84.07,170.34
2025-04-05,131.87,607.93,281.38,255.42,35.77,78.79,303.03,197.72,339.78,121.32,...,1065.83,648.44,823.34,5086.05,978.96,6438.14,57.82,107.80,84.07,170.34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-27,136.45,772.37,272.32,290.09,25.63,240.23,273.66,219.18,281.96,113.91,...,649.38,553.81,838.87,4779.03,849.10,6387.07,39.57,137.68,48.67,98.69
2025-12-28,135.83,826.40,275.21,293.47,28.11,242.56,275.97,231.44,309.34,126.12,...,662.80,555.54,818.53,4702.47,868.74,6933.94,70.10,127.55,55.77,113.11
2025-12-29,135.02,809.36,273.45,285.11,25.76,241.92,274.21,223.97,308.63,136.91,...,643.03,564.95,842.77,4806.47,842.39,6464.71,133.22,133.55,55.78,113.11
2025-12-30,129.40,769.54,268.86,287.73,25.71,227.76,269.88,218.30,271.64,125.70,...,671.68,556.62,804.64,4700.11,842.64,6421.54,132.61,130.19,55.78,113.12


### CREATING ADJACENCY MATRIX

In [26]:
G_raw = nx.DiGraph()
G_raw.add_edges_from(DATA_RAW.index.values)
G_raw = nx.line_graph(G_raw)


G_model = nx.Graph()
G_model.add_edges_from( [(EDGES_ID_DICT[x[0]], EDGES_ID_DICT[x[1]]) for x in G_raw.edges()] )

ADJ_M = nx.to_pandas_adjacency(G_model, dtype=int)
ADJ_M = ADJ_M.loc[DATA_RAW_MODEL.columns, DATA_RAW_MODEL.columns]
ADJ_M

,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,...,1112,1113,1114,1115,1116,1117,1118,1119,1120,1121
1000,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1001,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1002,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1003,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1004,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1117,0,0,0,0,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
1118,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1119,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1120,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### PARAMETERS

In [27]:
# TRAINING PARAMS
BATCH_SIZE = 32
EPOCHS = 60
LR = 0.001

# DATASET PARAMS
FREQ_H = int(re.search(r'[a-z]*(\d+)[a-z]*', SAMPLING)[1])
NUM_NODES = ADJ_M.shape[0]
HIST_STEPS = int(2*30*(24/FREQ_H))          # 2 months
PRED_STEPS = int(1*30*(24/FREQ_H))          # 1 month
INPUT_DIM = 1
OUTPUT_DIM = 1
D_MODEL = 64

STEPS_PER_DAY = int(24/FREQ_H)
DAYS_PER_WEEK = 7
MONTHS_PER_YEAR = 12
DAYS_PER_YEAR = 365

TIME_FEATURES = STEPS_PER_DAY + DAYS_PER_WEEK + MONTHS_PER_YEAR + DAYS_PER_YEAR


## CREATE DATASET

In [28]:
from node2vec import Node2Vec
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader

In [29]:
class DataProcessor:
    def __init__(self, df, history_steps=4, predict_steps=4, point_per_day=3):
        """
        df: DataFrame where Index=Timestamp (freq='3H'), Columns=Nodes
        history_steps: How many past steps to look at (e.g., 4 steps = 12 hours)
        predict_steps: How many future steps to predict (e.g., 4 steps = 12 hours)
        """
        self.df = df
        self.history_steps = history_steps
        self.predict_steps = predict_steps
        self.nodes = df.columns.tolist()
        self.num_nodes = len(self.nodes)
        
        # 3-Hour Freq means 8 steps per day
        self.point_per_day = point_per_day
        self.steps_per_day = int(24/point_per_day) 
        self.days_per_week = 7
        self.months_per_year = 12
        self.days_per_year = 365

    def get_time_features(self, timestamps):
        """
        Extracts Time-of-Day and Day-of-Week from timestamps.
        Returns: (Total_Samples, Steps_Per_Day + Days_Per_Week)
        """
        # 1. Time of Day (0 to 7 for 3H freq)
        # We calculate this by (Hour / point_per_day)
        tod = (timestamps.hour // self.point_per_day).values
        # One-hot encoding
        tod_onehot = np.eye(self.steps_per_day)[tod]
        
        # 2. Day of Week (0 to 6)
        dow = timestamps.dayofweek.values
        # One-hot encoding
        dow_onehot = np.eye(self.days_per_week)[dow]

        #3. day of year (1 to 365)
        doy = timestamps.dayofyear.values - 1
        doy_onehot = np.eye(self.days_per_year)[doy]

        # 4. Month (1 to 12)
        month = timestamps.month.values - 1
        month_onehot = np.eye(self.months_per_year)[month]
        
        # Concatenate: Shape (N,TF= 8 + 7 + 365 + 12)
        return np.concatenate([tod_onehot, dow_onehot, doy_onehot, month_onehot], axis=1)

    def create_dataset(self):
        # 1. Raw Data (Normalize if needed)
        data = self.df.values # (Total_Time, Nodes)
        timestamps = self.df.index
        
        # 2. Generate Time Features for the whole series
        time_features = self.get_time_features(timestamps) # (Total_Time, TF)
        
        X_list, Y_list = [], []
        TE_his_list, TE_pred_list = [], []
        
        # 3. Sliding Window
        total_len = len(data)
        window_size = self.history_steps + self.predict_steps
        
        for i in range(total_len - window_size + 1):
            # Indices
            hist_start = i
            hist_end = i + self.history_steps
            pred_end = hist_end + self.predict_steps
            
            # Data Slices
            # X: History Data (History, Nodes, 1)
            x_sample = data[hist_start:hist_end]
            x_sample = np.expand_dims(x_sample, axis=-1) # Add Feature Dim -> (H, N, 1)
            
            # Y: Target Data (Future, Nodes, 1)
            y_sample = data[hist_end:pred_end]
            y_sample = np.expand_dims(y_sample, axis=-1)
            
            # Time Embeddings
            # TE History: (History, 8+7+365+12)
            te_his_sample = time_features[hist_start:hist_end]
            
            # TE Future: (Future, 8+7+365+12)
            te_pred_sample = time_features[hist_end:pred_end]
            
            X_list.append(x_sample)
            Y_list.append(y_sample)
            TE_his_list.append(te_his_sample)
            TE_pred_list.append(te_pred_sample)
            
        # Convert to Tensor
        X = torch.FloatTensor(np.array(X_list))       # (B, H, N, 1)
        Y = torch.FloatTensor(np.array(Y_list))       # (B, P, N, 1)
        TE_his = torch.FloatTensor(np.array(TE_his_list)) # (B, H, TF)
        TE_pred = torch.FloatTensor(np.array(TE_pred_list)) # (B, P, TF)
        
        return TensorDataset(X, TE_his, TE_pred, Y)
'''
# TEST
processor = DataProcessor(DATA_RAW_MODEL, history_steps=HIST_STEPS, predict_steps=PRED_STEPS, point_per_day=FREQ_H)
train_dataset = processor.create_dataset()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

X, TE_his, TE_pred, Y_true = next(iter(train_loader))

print(X.shape)
print(TE_his.shape)
print(TE_pred.shape)
print(Y_true.shape)
'''

'\n# TEST\nprocessor = DataProcessor(DATA_RAW_MODEL, history_steps=HIST_STEPS, predict_steps=PRED_STEPS, point_per_day=FREQ_H)\ntrain_dataset = processor.create_dataset()\ntrain_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)\n\nX, TE_his, TE_pred, Y_true = next(iter(train_loader))\n\nprint(X.shape)\nprint(TE_his.shape)\nprint(TE_pred.shape)\nprint(Y_true.shape)\n'

In [30]:
#==========================================
# 1. NODE2VEC PRE-PROCESSING
# ==========================================
def generate_node2vec_embeddings(adj_m, embedding_dim=64):
    """
    Generates node embeddings using Node2Vec.
    adj_matrix: (N, N) numpy array (adjacency matrix)
    """
    adj_matrix = adj_m.to_numpy()


    # Create Graph
    rows, cols = np.where(adj_matrix > 0)
    G = nx.Graph() 
    for r, c in zip(rows, cols):
        G.add_edge(r, c, weight=adj_matrix[r, c])
        
    # Run Node2Vec
    # P=1, Q=1 is standard equivalent to DeepWalk, adjust for BFS/DFS bias
    node2vec = Node2Vec(G, dimensions=embedding_dim, walk_length=30, num_walks=200, workers=1, quiet=True)
    model = node2vec.fit(window=10, min_count=1, batch_words=4)
    
    # Extract vectors in order
    num_nodes = adj_matrix.shape[0]
    embeddings = np.zeros((num_nodes, embedding_dim))
    for i in range(num_nodes):
        if str(i) in model.wv:
            embeddings[i] = model.wv[str(i)]
        else:
            embeddings[i] = np.random.normal(0, 0.1, embedding_dim)
            
    return torch.tensor(embeddings, dtype=torch.float32)

#TEST
#node_vectors = generate_node2vec_embeddings(ADJ_M, embedding_dim=D_MODEL)
#print(f"Node Vectors Shape: {node_vectors.shape}")

In [31]:
# ==========================================
# 2. MODEL COMPONENTS
# ==========================================

class SpatioTemporalEmbedding(nn.Module):
    def __init__(self, node_vectors, d_model, day_dim=8, week_dim=7, month_dim=12, year_dim=365):
        super(SpatioTemporalEmbedding, self).__init__()
        num_nodes, input_emb_dim = node_vectors.shape
        
        # Spatial Embedding (Load Node2Vec)
        # freeze=False allows fine-tuning during training
        self.node_emb = nn.Embedding.from_pretrained(node_vectors, freeze=False)
        self.se_fc1 = nn.Linear(input_emb_dim, d_model)
        self.se_fc2 = nn.Linear(d_model, d_model)

        # Temporal Embedding (Time of Day + Day of Week)
        self.te_fc1 = nn.Linear(day_dim + week_dim + month_dim + year_dim, d_model)
        self.te_fc2 = nn.Linear(d_model, d_model)

    def forward(self, T_emb):
        """
        T_emb: (Batch, Time, Day+Week_Features+Month_Features+Year_Features)
        """
        # Spatial Path
        device = T_emb.device
        # Retrieve all node embeddings [0...N-1]
        node_indices = torch.arange(self.node_emb.num_embeddings, device=device)
        se = self.node_emb(node_indices)      
        se = F.relu(self.se_fc1(se))
        se = self.se_fc2(se) # (N, D)
        
        # Temporal Path
        te = F.relu(self.te_fc1(T_emb))
        te = self.te_fc2(te) # (B, T, D)
        
        return se, te

class STAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(STAttentionBlock, self).__init__()
        
        # Projections to mix X and STE before Attention
        self.spatial_proj = nn.Linear(2 * d_model, d_model)
        self.temporal_proj = nn.Linear(2 * d_model, d_model)

        # Built-in PyTorch Multihead Attention (Optimized)
        self.spatial_mha = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.temporal_mha = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)

        # Gated Fusion
        self.w_z1 = nn.Linear(d_model, d_model)
        self.w_z2 = nn.Linear(d_model, d_model)
        self.b_z = nn.Parameter(torch.zeros(d_model))

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, X, STE, mask=True):
        B, T, N, D = X.shape
        
        # --- 1. Spatial Attention (Nodes attend to Nodes) ---
        # Concatenate Input + STE
        X_ste = torch.cat([X, STE], dim=-1) # (B, T, N, 2D)
        
        # Prepare Query/Key/Value
        # Flatten Batch and Time to treat Nodes as sequence: (B*T, N, D)
        q_s = self.spatial_proj(X_ste).view(B*T, N, D)
        v_s = X.view(B*T, N, D)
        
        # MHA
        hs_out, _ = self.spatial_mha(q_s, q_s, v_s)
        HS = hs_out.view(B, T, N, D) # Restore shape

        # --- 2. Temporal Attention (Time attends to Time) ---
        # Permute to treat Time as sequence: (B*N, T, D)
        X_ste_t = X_ste.permute(0, 2, 1, 3).reshape(B*N, T, 2*D)
        v_t = X.permute(0, 2, 1, 3).reshape(B*N, T, D)
        
        q_t = self.temporal_proj(X_ste_t)
        
        # Causal Mask (Future cannot affect Past)
        attn_mask = None
        if mask:
            # Upper triangular mask with -inf
            attn_mask = torch.triu(torch.ones(T, T), diagonal=1).bool().to(X.device)

        ht_out, _ = self.temporal_mha(q_t, q_t, v_t, attn_mask=attn_mask, is_causal=mask)
        HT = ht_out.view(B, N, T, D).permute(0, 2, 1, 3) # Restore shape

        # --- 3. Gated Fusion ---
        z = torch.sigmoid(self.w_z1(HS) + self.w_z2(HT) + self.b_z)
        H = z * HS + (1 - z) * HT
        
        # Residual + Norm
        return self.norm1(X + H)

class TransformAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(TransformAttention, self).__init__()
        self.proj_q = nn.Linear(d_model, d_model)
        self.proj_k = nn.Linear(d_model, d_model)
        self.proj_v = nn.Linear(d_model, d_model)
        
        self.mha = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.out_linear = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, H_encoded, STE_his, STE_pred):
        B, P, N, D = H_encoded.shape
        Q = STE_pred.shape[1] # Future time steps
        
        # Flatten Batch and Nodes. Sequence is Time.
        # Query: Future STE (B*N, Q, D)
        query = self.proj_q(STE_pred).permute(0, 2, 1, 3).reshape(B*N, Q, D)
        
        # Key: Historical STE (B*N, P, D)
        key = self.proj_k(STE_his).permute(0, 2, 1, 3).reshape(B*N, P, D)
        
        # Value: Encoded History (B*N, P, D)
        value = self.proj_v(H_encoded).permute(0, 2, 1, 3).reshape(B*N, P, D)
        
        # Cross Attention (Future attends to Past)
        attn_out, _ = self.mha(query, key, value)
        
        # Reshape: (B*N, Q, D) -> (B, Q, N, D)
        context = attn_out.view(B, N, Q, D).permute(0, 2, 1, 3)
        
        return self.norm(self.out_linear(context))

class GMAN(nn.Module):
    def __init__(self, node_vectors, input_dim, output_dim, d_model=64, num_heads=4, num_layers=3, day_dim=STEPS_PER_DAY, week_dim=DAYS_PER_WEEK, month_dim=MONTHS_PER_YEAR, year_dim=DAYS_PER_YEAR):
        super(GMAN, self).__init__()
        
        # Embeddings
        self.ste = SpatioTemporalEmbedding(node_vectors, d_model, day_dim=day_dim, week_dim=week_dim, month_dim=month_dim, year_dim=year_dim)
        self.input_fc = nn.Linear(input_dim, d_model)
        
        # Encoder
        self.encoder_layers = nn.ModuleList([
            STAttentionBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        
        # Bridge
        self.transform_attn = TransformAttention(d_model, num_heads)
        
        # Decoder
        self.decoder_layers = nn.ModuleList([
            STAttentionBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        
        # Output
        self.output_fc_1 = nn.Linear(d_model, d_model)
        self.output_fc_2 = nn.Linear(d_model, output_dim)

    def forward(self, X, TE_his, TE_pred):
        """
        X: Historical Data (Batch, Hist_Steps, Nodes, Input_Dim)
        TE_his: History Time Features (Batch, Hist_Steps, Day+Week_Dim)
        TE_pred: Future Time Features (Batch, Pred_Steps, Day+Week_Dim)
        """
        # 1. Embeddings
        x_emb = F.relu(self.input_fc(X)) # (B, P, N, D)
        
        # Generate STE for History
        se, te_his = self.ste(TE_his) 
        # se is (N, D), te_his is (B, P, D). Broadcast se to match B and P
        ste_his = se.unsqueeze(0).unsqueeze(0) + te_his.unsqueeze(2) # (B, P, N, D)
        
        # Generate STE for Prediction
        _, te_pred = self.ste(TE_pred) # We reuse the same spatial embeddings
        ste_pred = se.unsqueeze(0).unsqueeze(0) + te_pred.unsqueeze(2) # (B, Q, N, D)
        
        # 2. Encoder
        h = x_emb
        for layer in self.encoder_layers:
            h = layer(h, ste_his, mask=True)
            
        # 3. Transform Attention (History -> Future)
        h_pred = self.transform_attn(h, ste_his, ste_pred)
        
        # 4. Decoder
        for layer in self.decoder_layers:
            # Mask=False in decoder because we already know the future STE 
            # (Note: In pure autoregressive tasks we mask, but GMAN translates STE to Traffic)
            h_pred = layer(h_pred, ste_pred, mask=False) 
            
        # 5. Output
        out = F.relu(self.output_fc_1(h_pred))
        out = self.output_fc_2(out) # (B, Q, N, Output_Dim)
        
        return out     
'''
STE = SpatioTemporalEmbedding(
    node_vectors, 
    D_MODEL,
    day_dim=STEPS_PER_DAY,
    week_dim=DAYS_PER_WEEK,
    month_dim=MONTHS_PER_YEAR,
    year_dim=DAYS_PER_YEAR
    )


se, te_his = STE(TE_his)
ste_his = se.unsqueeze(0).unsqueeze(0) + te_his.unsqueeze(2) # (B, P, N, D)

_, te_pred = STE(TE_pred) # We reuse the same spatial embeddings
ste_pred = se.unsqueeze(0).unsqueeze(0) + te_pred.unsqueeze(2) # (B, Q, N, D)

input_dim = 1
num_layers = 3



input_fc = nn.Linear(input_dim, D_MODEL)
x_emb = F.relu(input_fc(X))

encoder_layers = nn.ModuleList([
            STAttentionBlock(D_MODEL, num_heads=4, dropout=0.1) for _ in range(num_layers)
        ])

h = x_emb
for layer in encoder_layers:
    print(layer)
    h = layer(h, ste_his, mask=True)

'''


'\nSTE = SpatioTemporalEmbedding(\n    node_vectors, \n    D_MODEL,\n    day_dim=STEPS_PER_DAY,\n    week_dim=DAYS_PER_WEEK,\n    month_dim=MONTHS_PER_YEAR,\n    year_dim=DAYS_PER_YEAR\n    )\n\n\nse, te_his = STE(TE_his)\nste_his = se.unsqueeze(0).unsqueeze(0) + te_his.unsqueeze(2) # (B, P, N, D)\n\n_, te_pred = STE(TE_pred) # We reuse the same spatial embeddings\nste_pred = se.unsqueeze(0).unsqueeze(0) + te_pred.unsqueeze(2) # (B, Q, N, D)\n\ninput_dim = 1\nnum_layers = 3\n\n\n\ninput_fc = nn.Linear(input_dim, D_MODEL)\nx_emb = F.relu(input_fc(X))\n\nencoder_layers = nn.ModuleList([\n            STAttentionBlock(D_MODEL, num_heads=4, dropout=0.1) for _ in range(num_layers)\n        ])\n\nh = x_emb\nfor layer in encoder_layers:\n    print(layer)\n    h = layer(h, ste_his, mask=True)\n\n'

In [32]:
fffffffffffffffffffffffff

NameError: name 'fffffffffffffffffffffffff' is not defined

## TRAINING

In [33]:
processor = DataProcessor(DATA_RAW_MODEL, history_steps=HIST_STEPS, predict_steps=PRED_STEPS, point_per_day=FREQ_H)
DATA_DS = processor.create_dataset()
DATA_DL = DataLoader(DATA_DS, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

In [34]:
node_vectors = generate_node2vec_embeddings(ADJ_M, embedding_dim=D_MODEL)
print(f"Node Vectors Shape: {node_vectors.shape}")

Node Vectors Shape: torch.Size([122, 64])


In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Training on: {device}")

Training on: cuda


In [36]:
MODEL = GMAN(
        node_vectors=node_vectors,
        input_dim=1, 
        output_dim=1, 
        d_model=D_MODEL, 
        day_dim=STEPS_PER_DAY,
        week_dim=DAYS_PER_WEEK,
        month_dim=MONTHS_PER_YEAR,
        year_dim=DAYS_PER_YEAR
).to(device)

In [37]:
optimizer = torch.optim.Adam(MODEL.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

In [38]:
print("Starting Training...")
MODEL.train()
    
for epoch in range(10):
    total_loss = 0
    for batch_idx, (X, TE_his, TE_pred, Y) in enumerate(DATA_DL):
        # MOVE BATCH TO GPU
        X = X.to(device)
        TE_his = TE_his.to(device)
        TE_pred = TE_pred.to(device)
        Y = Y.to(device)
        
        optimizer.zero_grad()
        
        # Forward
        preds = MODEL(X, TE_his, TE_pred)
        
        # Loss
        loss = loss_fn(preds, Y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.5f}")

Starting Training...


OutOfMemoryError: CUDA out of memory. Tried to allocate 216.00 MiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 21.74 GiB is allocated by PyTorch, and 611.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
X, TE_his, TE_pred, Y = next(iter(DATA_DL))
X = X.to(device)
TE_his = TE_his.to(device)
TE_pred = TE_pred.to(device)
Y = Y.to(device)

print(f"X shape: {X.shape}")
print(f"TE_his shape: {TE_his.shape}")
print(f"TE_pred shape: {TE_pred.shape}")
print(f"Y shape: {Y.shape}")

X shape: torch.Size([32, 60, 122, 1])
TE_his shape: torch.Size([32, 60, 385])
TE_pred shape: torch.Size([32, 30, 385])
Y shape: torch.Size([32, 30, 122, 1])


In [ ]:
MODEL(X, TE_his, TE_pred)

tensor([[[[0.3595],
          [0.4776],
          [0.4055],
          ...,
          [0.3709],
          [0.3797],
          [0.4511]],

         [[0.3608],
          [0.4813],
          [0.3869],
          ...,
          [0.3618],
          [0.3798],
          [0.4332]],

         [[0.3676],
          [0.4739],
          [0.3986],
          ...,
          [0.3683],
          [0.3840],
          [0.4464]],

         ...,

         [[0.3694],
          [0.4736],
          [0.4004],
          ...,
          [0.3557],
          [0.3753],
          [0.4423]],

         [[0.3584],
          [0.4675],
          [0.4014],
          ...,
          [0.3517],
          [0.3856],
          [0.4398]],

         [[0.3719],
          [0.4814],
          [0.4191],
          ...,
          [0.3564],
          [0.4074],
          [0.4324]]],


        [[[0.3789],
          [0.4620],
          [0.3953],
          ...,
          [0.3829],
          [0.4172],
          [0.4554]],

         [[0.3652],
    

In [ ]:


# ==========================================
# 2. MODEL COMPONENTS
# ==========================================

###MALO
class SpatioTemporalEmbedding(nn.Module):
    def __init__(self, node_vectors, d_model, day_dim=288, week_dim=7):
        super(SpatioTemporalEmbedding, self).__init__()
        num_nodes, input_emb_dim = node_vectors.shape
        
        # Spatial Embedding (Load Node2Vec)
        # freeze=False allows fine-tuning during training
        self.node_emb = nn.Embedding.from_pretrained(node_vectors, freeze=False)
        self.se_fc1 = nn.Linear(input_emb_dim, d_model)
        self.se_fc2 = nn.Linear(d_model, d_model)

        # Temporal Embedding (Time of Day + Day of Week)
        self.te_fc1 = nn.Linear(day_dim + week_dim, d_model)
        self.te_fc2 = nn.Linear(d_model, d_model)

    def forward(self, T_emb):
        """
        T_emb: (Batch, Time, Day+Week_Features)
        """
        # Spatial Path
        device = T_emb.device
        # Retrieve all node embeddings [0...N-1]
        node_indices = torch.arange(self.node_emb.num_embeddings, device=device)
        se = self.node_emb(node_indices)      
        se = F.relu(self.se_fc1(se))
        se = self.se_fc2(se) # (N, D)
        
        # Temporal Path
        te = F.relu(self.te_fc1(T_emb))
        te = self.te_fc2(te) # (B, T, D)
        
        return se, te

### MALO

class STAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(STAttentionBlock, self).__init__()
        
        # Projections to mix X and STE before Attention
        self.spatial_proj = nn.Linear(2 * d_model, d_model)
        self.temporal_proj = nn.Linear(2 * d_model, d_model)

        # Built-in PyTorch Multihead Attention (Optimized)
        self.spatial_mha = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.temporal_mha = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)

        # Gated Fusion
        self.w_z1 = nn.Linear(d_model, d_model)
        self.w_z2 = nn.Linear(d_model, d_model)
        self.b_z = nn.Parameter(torch.zeros(d_model))

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, X, STE, mask=True):
        B, T, N, D = X.shape
        
        # --- 1. Spatial Attention (Nodes attend to Nodes) ---
        # Concatenate Input + STE
        X_ste = torch.cat([X, STE], dim=-1) # (B, T, N, 2D)
        
        # Prepare Query/Key/Value
        # Flatten Batch and Time to treat Nodes as sequence: (B*T, N, D)
        q_s = self.spatial_proj(X_ste).view(B*T, N, D)
        v_s = X.view(B*T, N, D)
        
        # MHA
        hs_out, _ = self.spatial_mha(q_s, q_s, v_s)
        HS = hs_out.view(B, T, N, D) # Restore shape

        # --- 2. Temporal Attention (Time attends to Time) ---
        # Permute to treat Time as sequence: (B*N, T, D)
        X_ste_t = X_ste.permute(0, 2, 1, 3).reshape(B*N, T, 2*D)
        v_t = X.permute(0, 2, 1, 3).reshape(B*N, T, D)
        
        q_t = self.temporal_proj(X_ste_t)
        
        # Causal Mask (Future cannot affect Past)
        attn_mask = None
        if mask:
            # Upper triangular mask with -inf
            attn_mask = torch.triu(torch.ones(T, T), diagonal=1).bool().to(X.device)

        ht_out, _ = self.temporal_mha(q_t, q_t, v_t, attn_mask=attn_mask, is_causal=mask)
        HT = ht_out.view(B, N, T, D).permute(0, 2, 1, 3) # Restore shape

        # --- 3. Gated Fusion ---
        z = torch.sigmoid(self.w_z1(HS) + self.w_z2(HT) + self.b_z)
        H = z * HS + (1 - z) * HT
        
        # Residual + Norm
        return self.norm1(X + H)


class TransformAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(TransformAttention, self).__init__()
        self.proj_q = nn.Linear(d_model, d_model)
        self.proj_k = nn.Linear(d_model, d_model)
        self.proj_v = nn.Linear(d_model, d_model)
        
        self.mha = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.out_linear = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, H_encoded, STE_his, STE_pred):
        B, P, N, D = H_encoded.shape
        Q = STE_pred.shape[1] # Future time steps
        
        # Flatten Batch and Nodes. Sequence is Time.
        # Query: Future STE (B*N, Q, D)
        query = self.proj_q(STE_pred).permute(0, 2, 1, 3).reshape(B*N, Q, D)
        
        # Key: Historical STE (B*N, P, D)
        key = self.proj_k(STE_his).permute(0, 2, 1, 3).reshape(B*N, P, D)
        
        # Value: Encoded History (B*N, P, D)
        value = self.proj_v(H_encoded).permute(0, 2, 1, 3).reshape(B*N, P, D)
        
        # Cross Attention (Future attends to Past)
        attn_out, _ = self.mha(query, key, value)
        
        # Reshape: (B*N, Q, D) -> (B, Q, N, D)
        context = attn_out.view(B, N, Q, D).permute(0, 2, 1, 3)
        
        return self.norm(self.out_linear(context))

In [ ]:
# ==========================================
# 3. MAIN GMAN MODEL
# ==========================================

class GMAN(nn.Module):
    def __init__(self, node_vectors, input_dim, output_dim, d_model=64, num_heads=4, num_layers=3):
        super(GMAN, self).__init__()
        
        # Embeddings
        self.ste = SpatioTemporalEmbedding(node_vectors, d_model)
        self.input_fc = nn.Linear(input_dim, d_model)
        
        # Encoder
        self.encoder_layers = nn.ModuleList([
            STAttentionBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        
        # Bridge
        self.transform_attn = TransformAttention(d_model, num_heads)
        
        # Decoder
        self.decoder_layers = nn.ModuleList([
            STAttentionBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        
        # Output
        self.output_fc_1 = nn.Linear(d_model, d_model)
        self.output_fc_2 = nn.Linear(d_model, output_dim)

    def forward(self, X, TE_his, TE_pred):
        """
        X: Historical Data (Batch, Hist_Steps, Nodes, Input_Dim)
        TE_his: History Time Features (Batch, Hist_Steps, Day+Week_Dim)
        TE_pred: Future Time Features (Batch, Pred_Steps, Day+Week_Dim)
        """
        # 1. Embeddings
        x_emb = F.relu(self.input_fc(X)) # (B, P, N, D)
        
        # Generate STE for History
        se, te_his = self.ste(TE_his) 
        # se is (N, D), te_his is (B, P, D). Broadcast se to match B and P
        ste_his = se.unsqueeze(0).unsqueeze(0) + te_his.unsqueeze(2) # (B, P, N, D)
        
        # Generate STE for Prediction
        _, te_pred = self.ste(TE_pred) # We reuse the same spatial embeddings
        ste_pred = se.unsqueeze(0).unsqueeze(0) + te_pred.unsqueeze(2) # (B, Q, N, D)
        
        # 2. Encoder
        h = x_emb
        for layer in self.encoder_layers:
            h = layer(h, ste_his, mask=True)
            
        # 3. Transform Attention (History -> Future)
        h_pred = self.transform_attn(h, ste_his, ste_pred)
        
        # 4. Decoder
        for layer in self.decoder_layers:
            # Mask=False in decoder because we already know the future STE 
            # (Note: In pure autoregressive tasks we mask, but GMAN translates STE to Traffic)
            h_pred = layer(h_pred, ste_pred, mask=False) 
            
        # 5. Output
        out = F.relu(self.output_fc_1(h_pred))
        out = self.output_fc_2(out) # (B, Q, N, Output_Dim)
        
        return out

# PARAMETERS

In [ ]:
ADJ_M.shape[0]

108

In [ ]:
ADJ_M

,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,...,1112,1113,1114,1115,1116,1117,1118,1119,1120,1121
1000,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1001,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1002,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1003,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1004,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1117,0,0,0,0,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
1118,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1119,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1120,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# 1. Simulate Graph & Node2Vec
node_vectors = generate_node2vec_embeddings(ADJ_M, embedding_dim=D_MODEL)
print(f"Node Vectors Shape: {node_vectors.shape}")

Node Vectors Shape: torch.Size([122, 64])


# TRAINING

In [ ]:
DATA_TS = DATA_RAW.copy()
DATA_TS.index = EDGES_ID['ID']
DATA_TS = DATA_TS.T

print(f"Shape of the time series: {DATA_TS.shape}")
DATA_TS.sample(10)

Shape of the time series: (2200, 122)


ID,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,...,1112,1113,1114,1115,1116,1117,1118,1119,1120,1121
2025-06-13 18:00:00,119.311805,530.698691,254.589072,242.920334,42.601650,186.040258,276.736424,184.689193,527.826409,99.599433,...,1276.310048,661.831965,831.739679,3668.503078,919.964716,7670.885675,35.243008,114.868873,99.549223,198.638854
2025-07-24 18:00:00,132.860718,858.021868,259.395908,246.400745,44.547175,186.040258,262.201265,211.128156,508.312554,134.858685,...,1.543237,672.857410,878.261133,4785.099878,1035.241298,6760.107342,47.302471,126.030223,177.501036,355.885946
2025-10-18 18:00:00,138.284039,552.574199,242.450476,270.489077,19.781587,235.787125,245.829414,220.142735,567.190542,142.503165,...,672.313053,708.115355,744.162324,5406.211208,918.557890,6985.448250,42.628999,131.796500,173.242858,277.510662
2025-04-21 03:00:00,86.850392,232.841873,170.469442,95.708994,9.652987,85.468895,180.385042,85.395900,117.343925,37.944764,...,550.619687,249.609697,328.053646,1654.337558,277.516279,2275.744858,17.712129,20.786339,78.814272,153.829506
2025-09-10 00:00:00,129.049918,793.312181,228.790959,229.808029,7.382565,158.832570,234.331551,199.556946,262.213604,120.016882,...,618.290076,660.064425,732.305719,3813.812116,1008.632361,6071.086647,52.214403,81.140643,149.353305,305.363889
2025-05-17 12:00:00,124.084965,708.801154,252.005792,218.262671,38.205774,168.298244,253.015373,152.900635,286.640266,92.817965,...,1209.358935,634.596539,759.046809,2282.734577,867.225774,7132.777356,30.955809,91.580722,202.164471,401.601388
2025-11-17 18:00:00,135.635239,271.536987,268.318927,275.689110,18.194040,240.056686,267.032514,217.812484,22.496557,107.553436,...,641.455872,642.272135,739.741280,5335.881890,980.277346,5804.677878,32.602510,149.969396,540.167301,1078.780467
2025-04-04 09:00:00,102.497043,528.081887,226.811301,160.497802,38.722708,142.086312,283.137380,88.360364,236.375522,93.360510,...,403.266161,573.117979,751.687427,4018.375682,668.359721,4513.109595,130.228793,82.539220,167.648699,325.898626
2025-11-01 21:00:00,136.633904,562.463125,262.077757,288.734070,27.672309,241.150706,263.218102,204.877817,549.929224,136.859369,...,631.490883,682.408254,822.539551,4834.807068,1051.168074,6893.560641,43.936722,146.510788,123.004547,290.027028
2025-10-23 03:00:00,35.113107,176.680994,100.503025,125.131899,11.792122,52.989970,96.075324,87.889947,104.929515,25.730156,...,245.887475,249.868796,266.238586,1063.062529,343.554698,1883.108602,15.106110,40.521156,21.876934,39.091728


### SOME STATISTICS

In [ ]:
print(f"The mean number of point by day is {DATA_TS.resample('D').count().iloc[:,0].mean().round().astype(int)}")
print(f"The mean number of point by week is {DATA_TS.resample('W').count().iloc[:,0].mean().round().astype(int)}")
print(f"The mean number of point by month is {DATA_TS.resample('ME').count().iloc[:,0].mean().round().astype(int)}")
print(f"The mean number of point by year is {DATA_TS.resample('YE').count().iloc[:,0].mean().round().astype(int)}")


The mean number of point by day is 8
The mean number of point by week is 55
The mean number of point by month is 244
The mean number of point by year is 2200


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [ ]:
NUM_NODES = 50
HIST_STEPS = 12
PRED_STEPS = 12
INPUT_DIM = 1
OUTPUT_DIM = 1
D_MODEL = 64
    
    # 1. Simulate Graph & Node2Vec
adj_matrix = np.random.randint(0, 2, (NUM_NODES, NUM_NODES))
adj_matrix

array([[1, 1, 1, ..., 0, 0, 1],
       [1, 1, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 0, 1, 1],
       ...,
       [0, 0, 0, ..., 1, 1, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 1, 1, 1]])

In [ ]:
# ==========================================
# 1. NODE2VEC PRE-PROCESSING
# ==========================================
def generate_node2vec_embeddings(adj_matrix, embedding_dim=64):
    """
    Generates node embeddings using Node2Vec.
    adj_matrix: (N, N) numpy array (adjacency matrix)
    """
    try:
        import networkx as nx
        from node2vec import Node2Vec
    except ImportError:
        print("Please install node2vec: pip install node2vec networkx")
        # Return random vectors for demonstration purposes if library missing
        return torch.randn(adj_matrix.shape[0], embedding_dim)

    # Create Graph
    rows, cols = np.where(adj_matrix > 0)
    G = nx.Graph() 
    for r, c in zip(rows, cols):
        G.add_edge(r, c, weight=adj_matrix[r, c])
        
    # Run Node2Vec
    # P=1, Q=1 is standard equivalent to DeepWalk, adjust for BFS/DFS bias
    node2vec = Node2Vec(G, dimensions=embedding_dim, walk_length=30, num_walks=200, workers=1, quiet=True)
    model = node2vec.fit(window=10, min_count=1, batch_words=4)
    
    # Extract vectors in order
    num_nodes = adj_matrix.shape[0]
    embeddings = np.zeros((num_nodes, embedding_dim))
    for i in range(num_nodes):
        if str(i) in model.wv:
            embeddings[i] = model.wv[str(i)]
        else:
            embeddings[i] = np.random.normal(0, 0.1, embedding_dim)
            
    return torch.tensor(embeddings, dtype=torch.float32)